# 💎 Jewelry Detection Model
## Complete CNN Model for Necklace and Ring Classification

This notebook will train a deep learning model to detect and classify jewelry images.

**Dataset**: Necklace and Ring images

**Steps**:
1. Upload dataset
2. Visualize data
3. Build CNN model
4. Train model
5. Evaluate results
6. Test predictions

**⚡ Enable GPU**: Runtime → Change runtime type → GPU

## Step 1: Import Libraries

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## Step 2: Upload Dataset
Upload your `archive.zip` file when prompted

In [ ]:
from google.colab import files

# Upload the archive.zip file
print("Please upload your 'archive.zip' file...")
uploaded = files.upload()

# Extract the zip file
zip_file_name = list(uploaded.keys())[0]
print(f"\nExtracting {zip_file_name}...")

with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
    zip_ref.extractall('.')

print("Extraction complete!")

## Step 3: Verify Data

In [ ]:
base_dir = 'Jewellery_Data'
categories = ['necklace', 'ring']

print("Dataset Information:")
print("="*50)
for category in categories:
    category_path = os.path.join(base_dir, category)
    num_images = len(os.listdir(category_path))
    print(f"{category.capitalize()}: {num_images} images")

## Step 4: Visualize Sample Images

In [ ]:
def display_sample_images(base_dir, categories, num_images=5):
    fig, axes = plt.subplots(len(categories), num_images, figsize=(15, 6))
    fig.suptitle('Sample Images from Dataset', fontsize=16, fontweight='bold')
    
    for i, category in enumerate(categories):
        category_path = os.path.join(base_dir, category)
        image_files = os.listdir(category_path)[:num_images]
        
        for j, img_file in enumerate(image_files):
            img_path = os.path.join(category_path, img_file)
            img = Image.open(img_path)
            
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
            if j == 0:
                axes[i, j].set_title(f'{category.capitalize()}', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

display_sample_images(base_dir, categories)

## Step 5: Prepare Data Generators

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

# Data Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Only rescaling for validation
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Create generators
train_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

validation_generator = val_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Classes: {train_generator.class_indices}")

## Step 6: Build CNN Model

In [ ]:
def create_model(num_classes):
    model = keras.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Block 4
        layers.Conv2D(256, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Block 5
        layers.Conv2D(512, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        # Dense layers
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Create and compile model
num_classes = len(categories)
model = create_model(num_classes)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 7: Setup Callbacks

In [ ]:
# Early stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Model checkpoint
checkpoint = ModelCheckpoint(
    'best_jewelry_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# Reduce learning rate
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

callbacks = [early_stopping, checkpoint, reduce_lr]

## Step 8: Train the Model
This may take 10-20 minutes with GPU

In [ ]:
EPOCHS = 50

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining Complete!")

## Step 9: Plot Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot accuracy
ax1.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot loss
ax2.plot(history.history['loss'], label='Training Loss', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10: Evaluate Model

In [ ]:
# Load best model
model = keras.models.load_model('best_jewelry_model.h5')

# Evaluate
val_loss, val_accuracy = model.evaluate(validation_generator, verbose=0)
print(f"\nValidation Accuracy: {val_accuracy*100:.2f}%")
print(f"Validation Loss: {val_loss:.4f}")

## Step 11: Confusion Matrix

In [ ]:
# Generate predictions
validation_generator.reset()
predictions = model.predict(validation_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = validation_generator.classes
class_labels = list(validation_generator.class_indices.keys())

# Confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Step 12: Classification Report

In [ ]:
print(classification_report(true_classes, predicted_classes, target_names=class_labels))

## Step 13: Test on Sample Images

In [ ]:
def predict_jewelry(image_path, model, class_labels):
    img = Image.open(image_path).resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    predictions = model.predict(img_array, verbose=0)
    predicted_class = class_labels[np.argmax(predictions)]
    confidence = np.max(predictions) * 100
    
    return predicted_class, confidence

# Test on random images
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

tested_images = 0
for category in categories:
    category_path = os.path.join(base_dir, category)
    image_files = os.listdir(category_path)
    
    for i in range(min(3, len(image_files))):
        if tested_images >= 6:
            break
            
        img_file = image_files[i]
        img_path = os.path.join(category_path, img_file)
        
        # Predict
        predicted_class, confidence = predict_jewelry(img_path, model, class_labels)
        
        # Display
        img = Image.open(img_path)
        axes[tested_images].imshow(img)
        axes[tested_images].axis('off')
        
        title_color = 'green' if predicted_class.lower() == category else 'red'
        axes[tested_images].set_title(
            f'True: {category}\nPredicted: {predicted_class}\nConfidence: {confidence:.1f}%',
            fontsize=10,
            color=title_color,
            fontweight='bold'
        )
        
        tested_images += 1

plt.tight_layout()
plt.show()

## Step 14: Save the Model

In [ ]:
# Save model
model.save('jewelry_detection_model.h5')
model.save('jewelry_detection_model_savedmodel')

print("Model saved successfully!")
print("- jewelry_detection_model.h5")
print("- jewelry_detection_model_savedmodel/")

## Step 15: Download Model (Optional)

In [ ]:
# Uncomment to download the model
# from google.colab import files
# files.download('best_jewelry_model.h5')

## 🎉 Training Complete!

Your jewelry detection model is ready!

### Next Steps:
1. ✅ Review the accuracy metrics
2. ✅ Check the confusion matrix
3. ✅ Test on sample images
4. ✅ Download the model
5. ✅ Use it in your applications!

### How to Use:
```python
# Load model
model = keras.models.load_model('best_jewelry_model.h5')

# Predict
img = Image.open('new_image.jpg').resize((224, 224))
img_array = np.array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)
prediction = model.predict(img_array)
```